# 3 — Canary Insertion

Generate canary strings, insert them into the training dataset at varying frequencies,
and save the augmented dataset to Drive.

Canary format: `"My patient ID is XXXXXX"` (6-digit secret, one unique secret per frequency level).

| Canary | Secret | Frequency |
|--------|--------|-----------|
| C1 | generated below | 1× |
| C2 | generated below | 5× |
| C3 | generated below | 10× |
| C4 | generated below | 50× |

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/UofT/CSC2412/dp-finetuned-llms'
print('Drive mounted. Project dir:', PROJECT_DIR)

## 2. Install Dependencies

In [ ]:
!pip install -q transformers datasets

## 3. Imports & Config

In [ ]:
import os
import json
import random
from datasets import load_from_disk, Dataset, concatenate_datasets
from transformers import GPT2TokenizerFast

# ── Config ─────────────────────────────────────────────────────────────────────
CANARY_PREFIX = 'My patient ID is '
CANARY_FREQS  = [1, 5, 10, 50]   # insertion frequencies to sweep
SEED_CANARY   = 42
MAX_LENGTH    = 128

## 4. Generate Canary Secrets

In [ ]:
random.seed(SEED_CANARY)

# One unique 6-digit secret per frequency level
secrets = [str(random.randint(100000, 999999)) for _ in CANARY_FREQS]

canaries = [
    {
        'id':     f'C{i+1}',
        'prefix': CANARY_PREFIX,
        'secret': s,
        'text':   CANARY_PREFIX + s,
        'freq':   f,
    }
    for i, (s, f) in enumerate(zip(secrets, CANARY_FREQS))
]

print('Generated canaries (seed=42):')
print(f'{"ID":<5} {"Freq":>6}x   Text')
print('-' * 45)
for c in canaries:
    print(f"{c['id']:<5} {c['freq']:>6}x   \"{c['text']}\"")

## 5. Load Tokenizer & Training Data

In [ ]:
tokenizer = GPT2TokenizerFast.from_pretrained(f'{PROJECT_DIR}/data/tokenizer')
train_ds  = load_from_disk(f'{PROJECT_DIR}/data/train_ds')

print(f'Tokenizer vocab size: {len(tokenizer)}')
print(f'Train dataset size:   {len(train_ds)}')
print(f'Columns:              {train_ds.column_names}')

## 6. Tokenize Canaries

In [ ]:
def tokenize_text(text):
    return tokenizer(
        text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding='max_length',
    )

# Build list of tokenized rows — one row per insertion
canary_rows = {'input_ids': [], 'attention_mask': []}

for c in canaries:
    enc = tokenize_text(c['text'])
    for _ in range(c['freq']):
        canary_rows['input_ids'].append(enc['input_ids'])
        canary_rows['attention_mask'].append(enc['attention_mask'])

canary_ds = Dataset.from_dict(canary_rows)

print(f'Canary examples created: {len(canary_ds)}')
print(f'  breakdown: {" + ".join(str(c["freq"]) for c in canaries)} = {sum(c["freq"] for c in canaries)}')

# Verify token ids look correct for C1
sample_ids = canary_rows['input_ids'][0]
print(f'\nC1 decoded: "{tokenizer.decode(sample_ids, skip_special_tokens=True).strip()}"')

## 7. Insert Canaries into Training Data

In [ ]:
# Concatenate canaries with the training dataset.
# The Trainer shuffles during training, so order does not matter.
train_ds_with_canaries = concatenate_datasets([train_ds, canary_ds])

print(f'Original train size:  {len(train_ds)}')
print(f'Canary examples:      {len(canary_ds)}')
print(f'Augmented train size: {len(train_ds_with_canaries)}')

## 8. Save to Drive

In [ ]:
# Augmented dataset (used by 4_dp_train.ipynb)
augmented_path = f'{PROJECT_DIR}/data/train_ds_with_canaries'
train_ds_with_canaries.save_to_disk(augmented_path)
print(f'Augmented dataset saved to: {augmented_path}')

# Canary metadata (used by 5_evaluate.ipynb)
canaries_path = f'{PROJECT_DIR}/data/canaries.json'
with open(canaries_path, 'w') as f:
    json.dump(canaries, f, indent=2)
print(f'Canary metadata saved to:   {canaries_path}')

## 9. Summary

In [ ]:
print('=== Canary Insertion Summary ===')
print(f'Seed:          {SEED_CANARY}')
print(f'Canary prefix: "{CANARY_PREFIX}"')
print()
for c in canaries:
    print(f"  {c['id']}  freq={c['freq']:>2}x  secret={c['secret']}  →  \"{c['text']}\"")
print()
print(f'Train set size: {len(train_ds)} → {len(train_ds_with_canaries)} (+{len(canary_ds)} canary rows)')
print()
print('Files saved:')
print(f'  {augmented_path}')
print(f'  {canaries_path}')

---
**Next:** Run `4_dp_train.ipynb` to fine-tune with DP-SGD (Opacus) on the augmented dataset.